# Evaluation and Error for CS74 Final Project: Who Does the Classifier Misread? Auditing Annotation Ambiguity in GoEmotions

Authored by Helen Cui 10/30/26

This notebook will cover evaluation and error analysis for GoEmotions which includes:
* ambiguity measures
* standard and soft-label metrics
* model comparison
* aggregation sensitivity
* slice analysis
* the qualitative coding scheme.

**Note:** This code will include utilizing the following files from `CS74_Final_Evaluation_Piping.ipynb` written by Piper Altman:
* predictions_*.parquet
* model_summary.csv
* thesholds_*.npy

This notebook also utilizes the raw GoEmotions data (one row per rater), loaded directly from Hugging Face

# Set Up & Configs

In [ ]:
import os, math, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import (f1_score, hamming_loss,
                             precision_recall_fscore_support, cohen_kappa_score)
from IPython.display import display

#Silence library warnings so printed tables stay readable
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120, "display.width", 200, "display.max_columns", 50)

# configurations
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/outputs"      # where data pipeline saved predictions_*.parquet, thresholds_*.npy, model_summary.csv
FIG, TAB = f"{OUT}/eval_figs", f"{OUT}/eval_tables"
for d in (OUT, FIG, TAB):
    os.makedirs(d, exist_ok=True)

SEED, N_BOOT = 42, 1000
TFIDF_TAG, EMB_TAG = None, None      # None = choose by validation macro-F1
ENTROPY_COL = "H_cat"                # primary ambiguity score (see definitions below); "H_bin" is the robustness variant
TOP_SUBREDDITS = 10                  # for the "for whom" analysis
SAMPLE_N = 40                        # qualitative review size
MIN_PAIR_OVERLAP = 50                # min shared comments for a rater pair to enter the pairwise Cohen's kappa
KAPPA_GAP_FLAG = 0.25                # flag emotions where raw agreement exceeds kappa by more than this
BINS = ["unanimous", "majority", "split"]
RES = {}                             # results registry used by the summary section


# helper methods
# Save a DataFrame as a CSV in the tables folder and return it unchanged,
# so it can be chained with display().
def save_table(df, name):
    df.to_csv(f"{TAB}/{name}.csv")
    return df

def save_fig(fig, name):
    fig.savefig(f"{FIG}/{name}.png", dpi=200, bbox_inches="tight")
print("OUT =", OUT)


Mounted at /content/drive
OUT = /content/drive/MyDrive/outputs


### Pick 2 audited models and load test predictions

Candidates are borrowed from part 1 of this project. Selection uses **validation** macro-F1 from `model_summary.csv`, produced by `CS74_Final_Evaluation_Piping.ipynb`

In [ ]:
# The variants exported by data pipeline, grouped by model family
TFIDF_CANDIDATES = ["a_word", "b_word_char", "c_word_char_hand", "a_word_casekept"]
EMB_CANDIDATES = ["d_embed", "e_word_plus_embed"]
pred_path = lambda tag: f"{OUT}/predictions_{tag}.parquet"
available = lambda tags: [t for t in tags if os.path.exists(pred_path(t))]

summary = pd.read_csv(f"{OUT}/model_summary.csv", index_col=0) if os.path.exists(f"{OUT}/model_summary.csv") else None


def pick(cands, override):
    '''  Choose which exported variant to audit for one model family.
    Inputs: candidate tags and an optional manual override.
    Returns the override if given, else the candidate with the best validation macro-F1
    (falls back to the first available file if model_summary.csv is missing).'''
    if override:
        return override

     # Only consider variants whose predictions file exists
    cands = available(cands)
    assert cands, f"No predictions_<tag>.parquet found in {OUT} for {cands}. Run Dataset Pipeline  first."
    if summary is not None:

        # Variants that have a validation macro-F1 in model_summary.csv
        scored = [c for c in cands if c in summary.index and pd.notna(summary.loc[c].get("val_macro_f1", np.nan))]
        if scored:
            return max(scored, key=lambda c: summary.loc[c, "val_macro_f1"])
    return cands[0]

# Choose the best variant per family for the rest of the notebook
TFIDF_TAG, EMB_TAG = pick(TFIDF_CANDIDATES, TFIDF_TAG), pick(EMB_CANDIDATES, EMB_TAG)
MODELS = {"tfidf": TFIDF_TAG, "emb": EMB_TAG}
SHORT = {"tfidf": "TF-IDF", "emb": "Embedding"}
COLORS = {"tfidf": "#1f77b4", "emb": "#d62728"}
print("Audited models:", MODELS)

# Read one prediction file to get the label names (column order) and the test comment ids
first = pd.read_parquet(pred_path(TFIDF_TAG))
names = [c[len("gold_"):] for c in first.columns if c.startswith("gold_")]
K = len(names)

# Test comments are the rows with split == 'test'; their order is the row order of every array below
test_ids = first.loc[first["split"] == "test", "comment_id"].tolist()
N = len(test_ids)
assert len(set(test_ids)) == N, "duplicate comment ids in test"

# P: probabilities, PRED: 0/1 predictions, TH: tuned thresholds, per model, on the test split
P, PRED, TH = {}, {}, {}


# Load each model's test rows in the shared comment order so rows line up across models
for m, tag in MODELS.items():
    d = pd.read_parquet(pred_path(tag))
    # same row order for both models
    d = d[d["split"] == "test"].set_index("comment_id").loc[test_ids]
    if m == "tfidf":
        Yg = d[[f"gold_{n}" for n in names]].to_numpy(np.int8)
    else:
        assert np.array_equal(Yg, d[[f"gold_{n}" for n in names]].to_numpy(np.int8)), "gold differs between models"
    P[m] = d[[f"prob_{n}" for n in names]].to_numpy(np.float32)
    PRED[m] = d[[f"pred_{n}" for n in names]].to_numpy(np.int8)
    thp = f"{OUT}/thresholds_{tag}.npy"
    TH[m] = np.load(thp) if os.path.exists(thp) else np.full(K, 0.5)
print(f"{N} test comments, {K} labels. Comments with an empty prediction: "
      + ", ".join(f"{SHORT[m]} {(PRED[m].sum(1) == 0).mean():.1%}" for m in MODELS))

Audited models: {'tfidf': 'c_word_char_hand', 'emb': 'e_word_plus_embed'}
5427 test comments, 28 labels. Comments with an empty prediction: TF-IDF 3.0%, Embedding 8.5%


### Load the raw (per-rater) GoEmotions data and collapse it to per-comment vote counts

The raw file has one row per (comment, rater) with one 0/1 column per emotion. For each comment sum over raters to get `votes[e]` (how many raters chose emotion `e`) and `n_raters`

In [ ]:
def load_raw():
    '''
    Load the raw GoEmotions data (one row per comment x rater) directly from Hugging Face.
    Falls back to Google's public CSV files if the Hub call fails.
    Returns a DataFrame with id, text, subreddit, rater_id, example_very_unclear and the 28 label columns.
    '''
    # Raw GoEmotions straight from Hugging Face (one row per comment x rater); nothing is read from or saved to Drive
    # Imported here so the rest of the notebook runs even if datasets is missing
    from datasets import load_dataset
    try:
        try:
            ds = load_dataset("google-research-datasets/go_emotions", "raw")
        except Exception:
            ds = load_dataset("go_emotions", "raw")
        raw = ds["train"].to_pandas()
    except Exception as e:
        print("Hugging Face raw config failed (", type(e).__name__, "), falling back to Google's CSVs")
        urls = [f"https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_{i}.csv" for i in (1, 2, 3)]
        raw = pd.concat([pd.read_csv(u) for u in urls], ignore_index=True)
    return raw[["id", "text", "subreddit", "rater_id", "example_very_unclear"] + names]

# Fetch the raw rater-level data
raw = load_raw()
assert set(names) <= set(raw.columns), set(names) - set(raw.columns)
print("raw rows (comment x rater):", len(raw), "| comments:", raw["id"].nunique(), "| raters:", raw["rater_id"].nunique())
print("rater rows with no emotion selected:", int((raw[names].sum(1) == 0).sum()),
      "| rows flagged example_very_unclear:", int(raw["example_very_unclear"].astype(bool).sum()), "(kept)")

# Collapse raters to one row per comment: emotion vote counts, number of raters, text and subreddit
grp = raw.groupby("id")
votes_all = grp[names].sum().astype(int)          # comment x emotion vote counts
nrat_all = grp.size()                             # raters per comment
meta_all = grp[["text", "subreddit"]].first()
# Every test comment must exist in the raw data, otherwise the join below is invalid
missing = set(test_ids) - set(votes_all.index)
assert not missing, f"{len(missing)} test comments not found in raw data"

# Test-set vote matrix and rater counts, aligned row-for-row with Yg, P and PRED
Vt = votes_all.loc[test_ids].to_numpy()           # test votes, aligned to Yg / P / PRED
nt = nrat_all.loc[test_ids].to_numpy()

# Share of raters who chose each emotion
S = Vt / nt[:, None]                              # per-emotion share of raters
print("raters per test comment:", pd.Series(nt).value_counts().sort_index().to_dict())

# Sanity check: the simplified gold labels should equal "emotion chosen by at least 2 raters"
# Rebuild the simplified gold labels (emotion chosen by at least 2 raters) from the raw votes
rebuilt = (Vt >= 2).astype(np.int8)
exact = (rebuilt == Yg).all(1).mean()
print(f"simplified gold == (votes >= 2) on {exact:.1%} of test comments")
if exact < 0.95:
    print("WARNING: the simplified labels do not match votes >= 2 for many comments; check the aggregation rule before trusting Section 6.")

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

raw/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 24.8MB            

raw/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/211225 [00:00<?, ? examples/s]

raw rows (comment x rater): 211225 | comments: 58011 | raters: 82
rater rows with no emotion selected: 3411 | rows flagged example_very_unclear: 0 (kept)
raters per test comment: {2: 52, 3: 3680, 4: 149, 5: 1546}
simplified gold == (votes >= 2) on 100.0% of test comments


### Load model_summary.csv

In [ ]:
# Standard metrics for the two audited models, read from model_summary.csv
assert summary is not None, "model_summary.csv not found in OUT; run Data Pipeline first"
std = summary.loc[[MODELS["tfidf"], MODELS["emb"]], ["macro_f1", "micro_f1", "hamming"]].copy()
std.index = [f"{SHORT[m]} ({MODELS[m]})" for m in MODELS]
display(std.round(4))
save_table(std, "01_standard_performance")
RES["std"] = std

# Consistency check: recompute from the exported predictions; each difference should be ~0
for m in MODELS:
    mine = [f1_score(Yg, PRED[m], average="macro", zero_division=0),
            f1_score(Yg, PRED[m], average="micro", zero_division=0),
            hamming_loss(Yg, PRED[m])]
    diff = np.abs(np.array(mine) - std.loc[f"{SHORT[m]} ({MODELS[m]})"].to_numpy()).max()
    print(f"{MODELS[m]}: max difference vs model_summary.csv = {diff:.5f}", "(OK)" if diff < 1e-4 else "<-- CHECK")


# No-text baselines from Dataset Pipeline , for reference
display(summary.reindex(["neutral_only", "label_frequency"])[["macro_f1", "micro_f1", "hamming"]].round(4))

,macro_f1,micro_f1,hamming
TF-IDF (c_word_char_hand),0.4715,0.5443,0.0462
Embedding (e_word_plus_embed),0.4381,0.5157,0.0483


c_word_char_hand: max difference vs model_summary.csv = 0.00000 (OK)
e_word_plus_embed: max difference vs model_summary.csv = 0.00000 (OK)


,macro_f1,micro_f1,hamming
neutral_only,0.0177,0.3040,0.0538
label_frequency,0.0438,0.1306,0.0730


In [ ]:
# Precision, recall, F1 and support for every label, per model
prf = {m: precision_recall_fscore_support(Yg, PRED[m], zero_division=0) for m in MODELS}


# Per-class table: support plus precision/recall/F1 for each model
per_class = pd.DataFrame({"support": Yg.sum(0)}, index=names)

for m in MODELS:
    per_class[f"P_{SHORT[m]}"], per_class[f"R_{SHORT[m]}"], per_class[f"F1_{SHORT[m]}"] = prf[m][0], prf[m][1], prf[m][2]


# Positive values mean the embedding model has the higher F1 for that class
per_class["F1_diff (Emb - TF-IDF)"] = per_class["F1_Embedding"] - per_class["F1_TF-IDF"]


# Most frequent classes first
per_class = per_class.sort_values("support", ascending=False)
display(per_class.round(3))
save_table(per_class, "01_per_class_performance")
RES["per_class"] = per_class

,support,P_TF-IDF,R_TF-IDF,F1_TF-IDF,P_Embedding,R_Embedding,F1_Embedding,F1_diff (Emb - TF-IDF)
neutral,1787,0.554,0.810,0.658,0.560,0.769,0.648,-0.009
admiration,504,0.611,0.683,0.645,0.559,0.639,0.596,-0.049
gratitude,352,0.911,0.903,0.907,0.850,0.838,0.844,-0.063
approval,351,0.249,0.399,0.306,0.275,0.419,0.332,0.025
annoyance,320,0.237,0.503,0.322,0.238,0.431,0.306,-0.016
curiosity,284,0.411,0.838,0.552,0.467,0.602,0.526,-0.025
disapproval,267,0.217,0.566,0.313,0.216,0.640,0.323,0.009
amusement,264,0.679,0.913,0.779,0.734,0.795,0.764,-0.015
love,238,0.723,0.878,0.793,0.725,0.786,0.754,-0.039
anger,198,0.356,0.576,0.440,0.418,0.475,0.444,0.004


# Ambiguity measures from the raw data
**Per-comment entropy (primary, `H_cat`).** Pool all emotion selections by all raters for a comment, normalise to a distribution `p_e = votes_e / sum(votes)` over the 28 labels, and take `H = -sum p_e log2 p_e` (bits). `H = 0` means every selection went to one emotion; larger means selections were spread over more emotions.

**Agreement bins** (from the share `s_top` of raters that chose the most-chosen emotion):
* *unanimous*: `s_top = 1` (every rater chose it)
* *majority*: `0.5 < s_top < 1`
* *split*: `s_top <= 0.5`

In [ ]:
def xlogx(p):
  '''Elementwise p * log2(p), defined as 0 where p == 0 (the usual entropy convention).'''
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(p > 0, p * np.log2(p), 0.0)

def ambiguity_measures(V, n):
  '''
  Per-comment disagreement measures from rater vote counts.
  Inputs: V (comments x emotions vote counts) and n (raters per comment).
  Returns a dict with H_cat (entropy of the pooled vote distribution), H_bin (sum of per-emotion
  binary entropies), top_share (share of raters choosing the most-chosen emotion) and agree_bin
  (unanimous / majority / split).
  '''
    # Work in floats; n becomes a column vector so it broadcasts across emotions
    V = np.asarray(V, float); n = np.asarray(n, float)[:, None]
    # p: each comment's pooled vote distribution over emotions
    p = V / np.maximum(V.sum(1, keepdims=True), 1)
    H_cat = -xlogx(p).sum(1) + 0.0                        # entropy of pooled vote distribution (+0.0 avoids printing -0.0)
    # s: share of raters choosing each emotion
    s = V / n
    H_bin = -(xlogx(s) + xlogx(1 - s)).sum(1) + 0.0       # sum of per-emotion binary entropies
    # Share of raters who chose the most-chosen emotion
    top = s.max(1)
    # Unanimous if every rater chose it, majority if more than half did, otherwise split
    b = np.where(top >= 1 - 1e-9, "unanimous", np.where(top > 0.5, "majority", "split"))
    return dict(H_cat=H_cat, H_bin=H_bin, top_share=top,
                agree_bin=pd.Categorical(b, categories=BINS, ordered=True))

# Compute entropy and agreement bin for every test comment
amb = ambiguity_measures(Vt, nt)
# D is the master table: one row per test comment; columns are added throughout the notebook
D = pd.DataFrame({"comment_id": test_ids,
                  "text": meta_all.loc[test_ids, "text"].to_numpy(),
                  "subreddit": meta_all.loc[test_ids, "subreddit"].to_numpy(),
                  "n_raters": nt, **amb})
# Number of gold labels per comment and comment length in words
D["n_gold"] = Yg.sum(1)
D["n_words"] = D["text"].str.split().str.len()

# Size and average disagreement of each agreement bin
bin_tbl = (D.groupby("agree_bin", observed=True)
             .agg(n=("comment_id", "size"), mean_entropy=("H_cat", "mean"), mean_H_bin=("H_bin", "mean"),
                  mean_n_gold=("n_gold", "mean"), mean_n_raters=("n_raters", "mean")))
bin_tbl.insert(1, "share", bin_tbl["n"] / N)
display(bin_tbl.round(3))
display(pd.crosstab(D["agree_bin"], D["n_raters"], margins=True))
save_table(bin_tbl, "02_agreement_bins")

# Left: histogram of entropy. Right: number of comments per agreement bin
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].hist(D["H_cat"], bins=30, color="gray"); ax[0].set_xlabel("per-comment entropy H_cat (bits)"); ax[0].set_ylabel("comments")
ax[1].bar(bin_tbl.index.astype(str), bin_tbl["n"], color="gray"); ax[1].set_ylabel("comments"); ax[1].set_xlabel("agreement bin")
fig.tight_layout(); save_fig(fig, "02_entropy_and_bins"); plt.show()

IndentationError: unexpected indent (5158534.py, line 3)

# Per-emotion inter-annotator agreement
For every emotion I look at all pairs of raters that labelled the same comment and ask whether both made the same yes/no decision on that emotion. Three numbers per emotion:
* **raw agreement**: share of rater pairs that made the same decision (including "both said no")
* **pooled kappa**: chance-corrected agreement using the pooled selection rate (equivalent to Scott's pi, since raters differ between comments)
* **pairwise Cohen's kappa**: classic Cohen's kappa computed for each specific rater pair with at least `MIN_PAIR_OVERLAP` shared comments, then averaged weighted by overlap size

I also report positive agreement (`2*both_yes / (2*both_yes + one_yes)`), which ignores the "both said no" cells that dominate for rare emotions. Computed on the full raw dataset (all comments) for stable estimates.

In [ ]:
def pairwise_iaa(raw, names, min_overlap=50):
  '''
  Per-emotion inter-annotator agreement from the raw rater-level data.
  Builds every pair of raters who labelled the same comment and, for each emotion, reports raw agreement,
  positive agreement, pooled kappa and the overlap-weighted average of per-rater-pair Cohen's kappas
  (only rater pairs with at least min_overlap shared comments).
  Returns (table indexed by emotion, number of distinct rater pairs, number of pair observations).
  '''
    # Work table: one row per (comment, rater) with a row index and an integer code per rater
    r = raw[["id", "rater_id"]].reset_index(drop=True)
    r["row"] = np.arange(len(r))
    r["rc"] = pd.factorize(r["rater_id"])[0]

    # L: boolean matrix of each rater's emotion selections
    L = raw[names].to_numpy(np.int8).astype(bool)

    # Self-join on comment id: every ordered pair of raters who rated the same comment
    m = r.merge(r, on="id", suffixes=("_a", "_b"))

    # Keep each unordered rater pair once per comment
    m = m[m["rc_a"] < m["rc_b"]]                                  # each rater pair on a comment once

    # A and B: the two raters' selections for every pair
    A, B = L[m["row_a"].to_numpy()], L[m["row_b"].to_numpy()]

    # Encode each rater pair as a single integer id so counts can be pooled per pair
    key = m["rc_a"].to_numpy().astype(np.int64) * (int(r["rc"].max()) + 1) + m["rc_b"].to_numpy()

    # uk: distinct rater pairs; inv: which pair each observation belongs to
    uk, inv = np.unique(key, return_inverse=True)

    # Sparse indicator matrix: multiplying by it sums observations within each rater pair
    G = sp.csr_matrix((np.ones(len(inv), dtype=np.float32), (inv, np.arange(len(inv)))), shape=(len(uk), len(inv)))

    # 2x2 agreement counts per rater pair and emotion: both chose it, only A chose it, only B chose it
    n11, n10, n01 = (G @ x.astype(np.float32) for x in (A & B, A & ~B, ~A & B))

    # n: number of shared comments for each rater pair
    n = np.asarray(G.sum(1)).ravel()[:, None]

    # Neither rater chose the emotion
    n00 = n - n11 - n10 - n01

    # Totals over all rater pairs, used for the pooled statistics
    T11, T10, T01, T00 = (x.sum(0) for x in (n11, n10, n01, n00)); Nt = n.sum()

    # po: observed agreement (both chose or both did not choose)
    po = (T11 + T00) / Nt

    # p1: overall rate at which the emotion is chosen
    p1 = (2 * T11 + T10 + T01) / (2 * Nt)

    # pe: agreement expected by chance at that rate
    pe = p1 ** 2 + (1 - p1) ** 2

    # Pooled kappa: chance-corrected agreement across all pairs
    kappa_pooled = (po - pe) / (1 - pe)

    # Each rater's own selection rate within a pair, for the classic Cohen's kappa
    pa, pb = (n11 + n10) / n, (n11 + n01) / n

    # Chance agreement and observed agreement for each individual rater pair
    pe_p = pa * pb + (1 - pa) * (1 - pb)
    po_p = (n11 + n00) / n

    # Only rater pairs with enough shared comments and a defined kappa are used
    valid = (n >= min_overlap) & (pe_p < 1 - 1e-9)

    # Cohen's kappa for each valid rater pair
    kp = np.where(valid, (po_p - pe_p) / np.maximum(1 - pe_p, 1e-9), 0.0)

    # Weight each pair by its number of shared comments
    w = np.where(valid, n, 0.0)

    # Overlap-weighted average of the pairwise kappas (NaN if no pair qualifies)
    kappa_pair = np.where(w.sum(0) > 0, (kp * w).sum(0) / np.maximum(w.sum(0), 1), np.nan)

    # Assemble the per-emotion results; positive agreement ignores the 'both said no' cell
    out = pd.DataFrame({"prevalence": p1, "raw_agreement": po, "pos_agreement": 2 * T11 / np.maximum(2 * T11 + T10 + T01, 1),
                        "kappa_pooled": kappa_pooled, "kappa_pairwise": kappa_pair,
                        "n_rater_pairs_used": valid.sum(0)}, index=names)

    # Gap between raw agreement and kappa; large for rare emotions
    out["agree_minus_kappa"] = out["raw_agreement"] - out["kappa_pooled"]
    return out, int(len(uk)), int(len(m))

# Run on the full raw dataset (all comments) for stable estimates
iaa, n_pairs_unique, n_pair_obs = pairwise_iaa(raw, names, MIN_PAIR_OVERLAP)
print(f"{n_pair_obs:,} rater-pair observations across {n_pairs_unique:,} distinct rater pairs")
iaa = iaa.sort_values("kappa_pooled", ascending=False)
display(iaa.round(3))
save_table(iaa, "02_per_emotion_agreement")
RES["iaa"] = iaa

# Emotions where raw agreement is much higher than kappa
flag = iaa[iaa["agree_minus_kappa"] > KAPPA_GAP_FLAG]
print(f"\nEmotions where raw agreement exceeds kappa by more than {KAPPA_GAP_FLAG}: {len(flag)} of {K}")
print(flag[["prevalence", "raw_agreement", "pos_agreement", "kappa_pooled"]].round(3).to_string() if len(flag) else "(none)")


# Check whether the gap between raw agreement and kappa tracks how rare the emotion is
rho_gap = stats.spearmanr(iaa["prevalence"], iaa["agree_minus_kappa"])[0]
print(f"\nSpearman(prevalence, raw agreement - kappa) across emotions = {rho_gap:+.2f}  (strongly negative supports the explanation below)")
print("Lowest kappa:", ", ".join(iaa.nsmallest(5, 'kappa_pooled').index), "| highest kappa:", ", ".join(iaa.nlargest(5, 'kappa_pooled').index))

# Error v. ambiguity

For each model: error rate by agreement bin, and the correlation between per-comment entropy and per-comment error, both with bootstrap 95% CIs.

Setup for the bootstrap: one shared matrix of resampled test-comment indices (`IDX`) is used for every statistic below, so differences between models, bins or analyses are paired.

In [ ]:
# Shared bootstrap resamples: each row is one resample of test-comment indices, reused by every statistic so comparisons are paired
IDX = np.random.default_rng(SEED).integers(0, N, size=(N_BOOT, N), dtype=np.int32)

def boot_ratio(x, mask):
  '''
  Bootstrap distribution of the mean of x over the rows where mask is True.
  Uses the shared resample matrix IDX, so results from different calls are paired.
  Returns an array of N_BOOT values.
  '''
    # bootstrap distribution of mean(x | mask)
    # Convert to float arrays so masks can be used as 0/1 weights
    x = np.asarray(x, float); mk = np.asarray(mask, float)
    # For each resample: sum of x over masked rows and the number of masked rows
    num = (x * mk)[IDX].sum(1); den = mk[IDX].sum(1)
    # Mean per resample; NaN when a resample contains no masked rows
    return num / np.where(den > 0, den, np.nan)


def ci95(a):
  '''
  95% percentile confidence interval of a bootstrap distribution, ignoring NaNs.
  Returns (lower, upper).
  '''
    lo, hi = np.nanpercentile(a, [2.5, 97.5]); return float(lo), float(hi)

def rate_ci(x, mask):
  '''Point estimate and bootstrap 95% CI of the mean of x over the masked rows.
  Returns (estimate, lower, upper); all NaN if the mask is empty.'''
    mask = np.asarray(mask, bool)
    # Nothing to estimate for an empty group
    if mask.sum() == 0:
        return np.nan, np.nan, np.nan
    lo, hi = ci95(boot_ratio(x, mask)); return float(np.asarray(x, float)[mask].mean()), lo, hi


def boot_spearman(x, y):
  '''
  Spearman correlation between x and y with a bootstrap 95% CI (resampling comments).
  Returns (rho, lower, upper).
  '''
    x, y = np.asarray(x, float), np.asarray(y, float)
    # Point estimate on the full test set
    rho = stats.spearmanr(x, y)[0]
    # Recompute the correlation on each bootstrap resample
    bs = np.array([stats.spearmanr(x[ii], y[ii])[0] for ii in IDX])
    return float(rho), *ci95(bs)

def boot_micro_f1(tp, fp, fn, mask):
  '''
  Bootstrap distribution of micro-F1 over the masked comments, from per-comment
  true positive / false positive / false negative counts. Returns an array of N_BOOT values.
  '''
    mk = np.asarray(mask, float)
    # Summed counts per resample (micro-F1 pools counts before dividing)
    TP, FP, FN = ((a * mk)[IDX].sum(1) for a in (tp, fp, fn))
    return 2 * TP / np.maximum(2 * TP + FP + FN, 1)

def per_comment_errors(Yg_, pr):
  '''
  Per-comment error measures for one model.
  Inputs: gold and predicted 0/1 matrices.
  Returns tp, fp, fn counts, wrong (1 if the predicted label set differs from gold at all),
  err_f1 (1 minus the comment's F1) and hamming (wrong labels / number of labels).

  '''
    # Use a wider integer type so the bitwise operations below are safe
    Yg_, pr = Yg_.astype(np.int16), pr.astype(np.int16)
    # Per comment: correctly predicted labels, extra labels, and missed labels
    tp = (Yg_ & pr).sum(1); fp = ((1 - Yg_) & pr).sum(1); fn = (Yg_ & (1 - pr)).sum(1)
    # Comment-level F1 is 2tp / (2tp + fp + fn); a comment with nothing to score counts as perfect
    den = 2 * tp + fp + fn
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 1.0)
    # wrong is 1 for any mismatch; err_f1 is 1 minus the comment F1
    return dict(tp=tp, fp=fp, fn=fn, wrong=((fp + fn) > 0).astype(int), err_f1=1 - f1, hamming=(fp + fn) / Yg_.shape[1])

# Per-comment error measures for each model, copied into D below
ERR = {m: per_comment_errors(Yg, PRED[m]) for m in MODELS}
for m in MODELS:
    for k in ("wrong", "err_f1", "tp", "fp", "fn"):
        D[f"{k}_{m}"] = ERR[m][k]


def entropy_groups(H, q=5):
  '''
  Split comments into q groups by entropy using quantile edges.
  Ties (many comments have entropy 0) fall into the first group, so fewer than q groups may result.
  Returns a Series of intervals.
  '''
    # Quantile edges; duplicates collapse because many comments share entropy 0
    edges = np.unique(np.quantile(H, np.linspace(0, 1, q + 1)))
    # Fall back to a median split if almost all values are identical
    if len(edges) < 3:
        edges = np.array([H.min() - 1e-9, np.median(H), H.max() + 1e-9])
    return pd.Series(pd.cut(H, edges, include_lowest=True))

# Entropy quintiles (and deciles, used later for the subreddit adjustment) to group comments
g5 = entropy_groups(D[ENTROPY_COL].to_numpy(), 5)
H_LEVELS = [str(c) for c in g5.cat.categories]
D["H_group"] = g5.astype(str).to_numpy()
D["H_decile"] = entropy_groups(D[ENTROPY_COL].to_numpy(), 10).astype(str).to_numpy()
print("entropy groups:", H_LEVELS, "| sizes:", D["H_group"].value_counts().reindex(H_LEVELS).tolist())

In [ ]:
# Error rate by agreement bin, with bootstrap CIs
rows = []
for m in MODELS:
    for b in BINS:
        mk = (D["agree_bin"] == b).to_numpy()
        w, wlo, whi = rate_ci(D[f"wrong_{m}"], mk)
        e, elo, ehi = rate_ci(D[f"err_f1_{m}"], mk)
        rows.append(dict(model=SHORT[m], agree_bin=b, n=int(mk.sum()), error_rate=w, er_lo=wlo, er_hi=whi,
                         mean_err_f1=e, ef_lo=elo, ef_hi=ehi))
# Error rate and mean 1-F1 per agreement bin, each with a bootstrap CI
err_by_bin = pd.DataFrame(rows)
display(err_by_bin.round(3))
save_table(err_by_bin.set_index(["model", "agree_bin"]), "03_error_by_agreement_bin")
RES["err_by_bin"] = err_by_bin

# Difference between bins (paired bootstrap): is 'split' worse than 'unanimous'?
for m in MODELS:
    d = boot_ratio(D[f"wrong_{m}"], D["agree_bin"] == "split") - boot_ratio(D[f"wrong_{m}"], D["agree_bin"] == "unanimous")
    pt = D.loc[D["agree_bin"] == "split", f"wrong_{m}"].mean() - D.loc[D["agree_bin"] == "unanimous", f"wrong_{m}"].mean()
    lo, hi = ci95(d)
    print(f"{SHORT[m]}: error(split) - error(unanimous) = {pt:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")

In [ ]:
# Correlation between entropy and per-comment error (Spearman), with bootstrap CIs
rows = []
for m in MODELS:
    for ent in ("H_cat", "H_bin"):
        for err in ("err_f1", "wrong"):
            rho, lo, hi = boot_spearman(D[ent], D[f"{err}_{m}"])
            rows.append(dict(model=SHORT[m], entropy=ent, error=err, spearman=rho, ci_lo=lo, ci_hi=hi,
                             primary=(ent == ENTROPY_COL)))
# Spearman correlation between entropy and error, for both entropy definitions and both error measures
corr_tbl = pd.DataFrame(rows)
display(corr_tbl.round(3))
save_table(corr_tbl.set_index(["model", "entropy", "error"]), "03_entropy_error_correlation")
RES["corr"] = corr_tbl

In [ ]:
# Is the relationship just a side effect of label cardinality or rater count?  Error by bin, split by n_gold and n_raters.
# Control: does the bin effect survive when the number of gold labels is held fixed?
D["gold_card"] = np.where(D["n_gold"] == 1, "1 gold label", "2+ gold labels")
tabs = []
for m in MODELS:
    t1 = D.groupby(["gold_card", "agree_bin"], observed=True)[f"wrong_{m}"].agg(["mean", "size"]).rename(columns={"mean": f"err_{SHORT[m]}", "size": f"n_{SHORT[m]}"})
    tabs.append(t1)
strat = pd.concat(tabs, axis=1).loc[:, lambda x: ~x.columns.duplicated()]
display(strat.round(3))

# The same check split by number of raters per comment
nr_tab = pd.concat({SHORT[m]: D.groupby(["n_raters", "agree_bin"], observed=True)[f"wrong_{m}"].mean().unstack() for m in MODELS}, axis=1)
display(nr_tab.round(3))
save_table(strat, "03_error_by_bin_stratified_gold_cardinality")

In [ ]:
# Figure: error vs ambiguity, one line per model (deliverable)
# Left: error by entropy quintile, one line per model with CI bars. Right: error by agreement bin
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

for m in MODELS:
    xs, ys, lo, hi = [], [], [], []
    # Point estimate and CI of the error rate within each entropy quintile
    for lv in H_LEVELS:
        mk = (D["H_group"] == lv).to_numpy()
        pt, l, h = rate_ci(D[f"wrong_{m}"], mk)
        xs.append(D.loc[mk, ENTROPY_COL].mean()); ys.append(pt); lo.append(l); hi.append(h)
    ys, lo, hi = np.array(ys), np.array(lo), np.array(hi)
    ax[0].errorbar(xs, ys, yerr=[ys - lo, hi - ys], marker="o", capsize=3, color=COLORS[m], label=SHORT[m])

ax[0].set_xlabel(f"mean per-comment entropy ({ENTROPY_COL}, bits) within entropy quintile")
ax[0].set_ylabel("error rate (predicted label set != gold)"); ax[0].set_title("Error vs. ambiguity"); ax[0].legend()

for m in MODELS:
    t = err_by_bin[err_by_bin.model == SHORT[m]]
    ax[1].errorbar(range(len(BINS)), t.error_rate, yerr=[t.error_rate - t.er_lo, t.er_hi - t.error_rate],
                   marker="o", capsize=3, color=COLORS[m], label=SHORT[m])

ax[1].set_xticks(range(len(BINS))); ax[1].set_xticklabels(BINS); ax[1].set_xlabel("agreement bin"); ax[1].set_title("Error by agreement bin")
fig.tight_layout(); save_fig(fig, "03_error_vs_ambiguity"); plt.show()

# Model comparison between embedding and TF-IDF
Each test comment is assigned to one pattern: both models miss it, only TF-IDF misses it (the embedding model fixes it), only the embedding model misses it, or neither misses it. "Miss" means `wrong = 1` (predicted label set differs from gold). The comparison statistic is `Delta = error(TF-IDF) - error(Embedding)`, so a positive Delta means the embedding model is better. Hypothesis: Delta is concentrated in low-disagreement comments.

In [ ]:
# Binary 'wrong' flags for each model (1 = predicted label set differs from gold)
wt, we = D["wrong_tfidf"].to_numpy(), D["wrong_emb"].to_numpy()
PATTERNS = ["both_miss", "only_tfidf_miss", "only_emb_miss", "neither_miss"]
# Label each comment: both wrong, only TF-IDF wrong, only the embedding model wrong, or neither
D["pattern"] = np.select([(wt == 1) & (we == 1), (wt == 1) & (we == 0), (wt == 0) & (we == 1)],
                         PATTERNS[:3], PATTERNS[3])
# Counts of each pattern within each agreement bin
ct = pd.crosstab(D["agree_bin"], D["pattern"]).reindex(columns=PATTERNS, fill_value=0)
display(ct)
display((ct.div(ct.sum(1), axis=0)).round(3))
save_table(ct, "04_pattern_by_agreement_bin")


def delta_table(col, levels):
  '''
  Compare the two models within each level of a grouping column.
  Delta = error(TF-IDF) - error(Embedding), so a positive value means the embedding model is better.
  Reports bootstrap CIs and an exact McNemar p-value on the discordant comments.
  '''
    # Per comment: +1 if only TF-IDF is wrong, -1 if only the embedding model is wrong, else 0
    d = wt - we
    rows = []
    for lv in levels:
        # Comments in this level of the grouping column
        mk = (D[col].astype(str) == lv).to_numpy()
        if mk.sum() == 0:
            continue
        # Mean gain and its bootstrap CI
        pt = d[mk].mean(); lo, hi = ci95(boot_ratio(d, mk))
        # Discordant counts: only TF-IDF wrong (b) and only the embedding model wrong (c)
        b = int(((wt == 1) & (we == 0) & mk).sum()); c = int(((wt == 0) & (we == 1) & mk).sum())
        # Exact McNemar test: are b and c equally likely?
        pval = stats.binomtest(b, b + c, 0.5).pvalue if (b + c) > 0 else np.nan     # exact McNemar
        rows.append(dict(level=lv, n=int(mk.sum()), err_tfidf=wt[mk].mean(), err_emb=we[mk].mean(), delta=pt,
                         lo=lo, hi=hi, only_tfidf_wrong=b, only_emb_wrong=c, mcnemar_p=pval))
    return pd.DataFrame(rows)

# Embedding gain by agreement bin and by entropy quintile
delta_bin = delta_table("agree_bin", BINS); delta_ent = delta_table("H_group", H_LEVELS)
print("Delta by agreement bin (positive = embedding better):"); display(delta_bin.round(4))
print("Delta by entropy quintile:"); display(delta_ent.round(4))
# Overall embedding gain with a bootstrap CI
overall = (wt - we).mean(); olo, ohi = ci95(boot_ratio(wt - we, np.ones(N, bool)))
print(f"Overall Delta = {overall:+.4f}  95% CI [{olo:+.4f}, {ohi:+.4f}]")

# The hypothesis test: is the embedding gain larger on unanimous than on split comments?
# Hypothesis test: embedding gain on unanimous minus gain on split comments, with a bootstrap CI
inter = boot_ratio(wt - we, D["agree_bin"] == "unanimous") - boot_ratio(wt - we, D["agree_bin"] == "split")
ipt = (wt - we)[(D["agree_bin"] == "unanimous").to_numpy()].mean() - (wt - we)[(D["agree_bin"] == "split").to_numpy()].mean()
ilo, ihi = ci95(inter)
print(f"Delta(unanimous) - Delta(split) = {ipt:+.4f}  95% CI [{ilo:+.4f}, {ihi:+.4f}]  ->",
      "supports 'gain concentrated on low-disagreement comments'" if ilo > 0 else
      ("opposite direction" if ihi < 0 else "CI includes 0: not distinguishable"))
save_table(delta_bin.set_index("level"), "04_delta_by_agreement_bin"); save_table(delta_ent.set_index("level"), "04_delta_by_entropy_group")
RES["delta_bin"], RES["delta_ent"] = delta_bin, delta_ent
RES["interaction"] = dict(est=ipt, lo=ilo, hi=ihi, overall=overall, overall_lo=olo, overall_hi=ohi)

In [ ]:
# Left: pattern shares per bin as stacked bars. Right: embedding gain per bin with CI
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
share = ct.div(ct.sum(1), axis=0)
bottom = np.zeros(len(share))
pal = {"both_miss": "#444444", "only_tfidf_miss": COLORS["tfidf"], "only_emb_miss": COLORS["emb"], "neither_miss": "#dddddd"}

for p in PATTERNS:
    ax[0].bar(share.index.astype(str), share[p], bottom=bottom, color=pal[p], label=p); bottom += share[p].to_numpy()
ax[0].set_ylabel("share of comments"); ax[0].set_title("Who misses what, by agreement bin"); ax[0].legend(fontsize=7, loc="lower right")
ax[1].bar(delta_bin["level"], delta_bin["delta"], yerr=[delta_bin["delta"] - delta_bin["lo"], delta_bin["hi"] - delta_bin["delta"]],
          capsize=4, color="gray"); ax[1].axhline(0, color="k", lw=0.8)

ax[1].set_ylabel("error(TF-IDF) - error(Embedding)"); ax[1].set_title("Embedding gain by agreement bin (>0: embedding better)")

fig.tight_layout(); save_fig(fig, "04_model_comparison"); plt.show()

## 5. Soft-label evaluation
### Jensen-Shannon divergence between predicted probabilities and rater vote distributions
Per comment, `JSD(q, p_model)` in bits, where `q` is the rater distribution and `p_model` is the model's 28 probabilities renormalised to sum to 1 (squared Jensen-Shannon *distance*, so it lies in [0, 1]). Two caveats: the models are one-vs-rest with class-balanced weights, so their probabilities are not calibrated (the scaled variant divides each class probability by its tuned threshold first), and rater distributions come from only 3 to 5 raters. As a yardstick, the prior row predicts the average rater distribution of the non-test comments for every comment.

In [ ]:
def jsd(Pm, Qm):
  '''
  Jensen-Shannon divergence in bits (squared JS distance, range 0 to 1) between each row of
  Pm and the matching row of Qm. Both must be probability distributions.
  '''
    # M: the average of the two distributions (the mixture)
    M = 0.5 * (Pm + Qm)
    # Kullback-Leibler divergence of a from b, summed over labels (terms with a == 0 contribute 0).
    def kl(a, b):
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.where(a > 0, a * np.log2(a / b), 0.0).sum(1)
    # JSD is the mean KL divergence of each distribution from the mixture
    return 0.5 * kl(Pm, M) + 0.5 * kl(Qm, M)

# Rescale each row to sum to 1 so it can be read as a distribution
renorm = lambda A: A / np.maximum(A.sum(1, keepdims=True), 1e-12)
# Q: the rater vote distribution of each comment
Q = renorm(Vt.astype(float))
# Reference 'prior': the average rater distribution over all non-test comments
rest = votes_all.drop(index=test_ids).to_numpy().sum(0).astype(float)
prior = (rest / rest.sum())[None, :]
# JSD of the prior against each comment's rater distribution
D["jsd_prior"] = jsd(np.repeat(prior, N, 0), Q)
for m in MODELS:
    # JSD of each model's renormalised probabilities (raw and threshold-scaled) against the rater distribution
    D[f"jsd_{m}"] = jsd(renorm(P[m].astype(float)), Q)
    D[f"jsd_scaled_{m}"] = jsd(renorm(P[m] / np.maximum(TH[m][None, :], 1e-6)), Q)

rows = []
for name, col in [(SHORT[m], f"jsd_{m}") for m in MODELS] + [(f"{SHORT[m]} (scaled by threshold)", f"jsd_scaled_{m}") for m in MODELS] + [("Prior (train average)", "jsd_prior")]:
    r = {"model": name}
    for b in BINS + ["all"]:
        mk = np.ones(N, bool) if b == "all" else (D["agree_bin"] == b).to_numpy()
        pt, lo, hi = rate_ci(D[col], mk); r[b] = f"{pt:.3f} [{lo:.3f}, {hi:.3f}]"
    r["spearman(entropy, JSD)"] = "{:+.2f} [{:+.2f}, {:+.2f}]".format(*boot_spearman(D[ENTROPY_COL], D[col]))
    rows.append(r)

# JSD by agreement bin with bootstrap CIs, plus its correlation with entropy
jsd_tbl = pd.DataFrame(rows).set_index("model")
display(jsd_tbl)
save_table(jsd_tbl, "05_jsd_by_agreement_bin"); RES["jsd"] = jsd_tbl

### Evaluation of training golds and false positives
The training gold keeps every emotion with two or more rater votes, so any extra (false-positive) emotion a model predicts has at most onerater vote by construction. The question therefore becomes: when the model is wrong, is its extra emotion something *one* rater chose (a minority reading), or something *no* rater chose? Each wrong comment falls in exactly one category:
* `omission_only`: nothing extra predicted; the model only missed gold labels
* `fp_unsupported`: at least one extra emotion predicted and no rater chose any of them
* `fp_one_rater`: at least one extra emotion predicted and a rater chose at least one of them

In [ ]:
# Categories for wrong predictions (see the text above)
cat_names = ["omission_only", "fp_unsupported", "fp_one_rater"]
for m in MODELS:
    FPm = (PRED[m] == 1) & (Yg == 0)
    n_fp = FPm.sum(1)
    # True when at least one rater chose an extra emotion the model predicted
    supp = (FPm & (Vt >= 1)).any(1)
    wrong_m = D[f"wrong_{m}"].to_numpy() == 1
    # Assign each comment to omission-only / unsupported extra emotion / extra emotion chosen by one rater
    cat = np.select([~wrong_m, n_fp == 0, ~supp], ["correct", "omission_only", "fp_unsupported"], "fp_one_rater")
    D[f"errcat_{m}"] = cat

rows = []
for m in MODELS:
    wr = D[D[f"wrong_{m}"] == 1]
    for b in BINS + ["all"]:
        sub = wr if b == "all" else wr[wr["agree_bin"] == b]
        row = dict(model=SHORT[m], agree_bin=b, n_errors=len(sub))
        for c in cat_names:
            row[c] = (sub[f"errcat_{m}"] == c).mean() if len(sub) else np.nan
        rows.append(row)
# Share of each category among the wrong predictions, by agreement bin
plaus = pd.DataFrame(rows)
display(plaus.round(3))
save_table(plaus.set_index(["model", "agree_bin"]), "05_wrong_predictions_rater_support")
RES["plaus"] = plaus

# Do model predictions track minority rater votes?  Predicted rate by number of raters who chose the label.
rows = []
for m in MODELS:
    for v in (0, 1, 2, 3):
        mk = (np.minimum(Vt, 3) == v)
        rows.append(dict(model=SHORT[m], raters_choosing_label=("3+" if v == 3 else str(v)), n_label_instances=int(mk.sum()),
                         predicted_rate=PRED[m][mk].mean()))
# Do the models predict a label more often when more raters chose it?
pred_by_votes = pd.DataFrame(rows).pivot(index="raters_choosing_label", columns="model", values="predicted_rate")
print("Share of (comment, emotion) pairs the model predicts, by how many raters chose that emotion:")
display(pred_by_votes.round(4))
save_table(pred_by_votes, "05_predicted_rate_by_votes")

In [ ]:
def wilson(k, n, z=1.96):
  '''
  # Recall of gold labels as a function of how strongly raters supported the label
# Wilson 95% confidence interval for a proportion k / n. Returns (lower, upper).
  '''
    if n == 0: return np.nan, np.nan
    # Observed proportion and the Wilson centre and half-width
    p = k / n; den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    # Interval limits
    return c - h, c + h

# Recall of gold labels by the share of raters who chose the label
gm = Yg == 1
share_g = S[gm]
# Bucket each gold label by rater share
sbin = pd.cut(share_g, [0, 0.4, 0.5, 0.7, 0.99, 1.0], labels=["<=0.40", "0.40-0.50", "0.50-0.70", "0.70-0.99", "1.00"])
rows = []
for m in MODELS:
    hit = PRED[m][gm]
    for lv in sbin.categories:
        mk = np.asarray(sbin == lv)
        k, n_ = int(hit[mk].sum()), int(mk.sum()); lo, hi = wilson(k, n_)
        rows.append(dict(model=SHORT[m], rater_share_for_label=lv, n=n_, recall=k / n_ if n_ else np.nan, lo=lo, hi=hi))
rec_share = pd.DataFrame(rows)
display(rec_share.pivot(index="rater_share_for_label", columns="model", values="recall").round(3))
save_table(rec_share.set_index(["model", "rater_share_for_label"]), "05_recall_by_rater_support")
RES["rec_share"] = rec_share

# Left: recall by rater support. Right: mean JSD by agreement bin (dashed = prior)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for m in MODELS:
    t = rec_share[rec_share.model == SHORT[m]]
    ax[0].errorbar(range(len(t)), t.recall, yerr=[t.recall - t.lo, t.hi - t.recall], marker="o", capsize=3, color=COLORS[m], label=SHORT[m])
ax[0].set_xticks(range(len(sbin.categories))); ax[0].set_xticklabels(sbin.categories); ax[0].set_xlabel("share of raters who chose the gold label"); ax[0].set_ylabel("recall"); ax[0].legend()
ax[0].set_title("Recall rises with rater support?")
for m in MODELS:
    t = D[D["jsd_" + m].notna()].groupby("agree_bin", observed=True)["jsd_" + m].mean()
    ax[1].plot(range(len(BINS)), t.reindex(BINS).to_numpy(), marker="o", color=COLORS[m], label=SHORT[m])
ax[1].plot(range(len(BINS)), D.groupby("agree_bin", observed=True)["jsd_prior"].mean().reindex(BINS).to_numpy(), "k--", marker="s", label="prior")
ax[1].set_xticks(range(len(BINS))); ax[1].set_xticklabels(BINS); ax[1].set_ylabel("mean JSD to rater distribution"); ax[1].legend(); ax[1].set_title("Soft-label divergence by agreement bin")
fig.tight_layout(); save_fig(fig, "05_soft_label"); plt.show()

## Aggregation sensitivity
Fixed predictions with only the gold definition changes. Gold sets are rebuilt from the raw rater votes:
* **simplified**: emotions with at least 2 rater votes (what the models were trained and thresholded on)
* **any rater**: at least 1 vote
* **majority**: more than 50% of raters
* **unanimous**: every rater

Per-class F1 is undefined for a class with no positives under a definition (common for rare emotions under unanimous); those classes are left out of that definition's macro-F1, and the number left out is shown.

In [ ]:
# Four gold definitions rebuilt from the raw votes; predictions stay fixed
GOLDS = {"simplified": Yg.astype(int),
         "any_rater": (Vt >= 1).astype(int),
         "majority": (S > 0.5).astype(int),
         "unanimous": (Vt == nt[:, None]).astype(int)}
rows, macro_rows = {}, []
# Score the fixed predictions against each gold definition
for gname, G in GOLDS.items():
    sup = G.sum(0)
    for m in MODELS:
        f = f1_score(G, PRED[m], average=None, zero_division=0).astype(float)
        # F1 is undefined for classes with no positives under this definition: leave them out of the macro average
        f[sup == 0] = np.nan
        rows[(SHORT[m], gname)] = pd.Series(f, index=names)
        macro_rows.append(dict(model=SHORT[m], gold=gname, macro_f1=np.nanmean(f), classes_scored=int((sup > 0).sum()),
                               classes_dropped=int((sup == 0).sum()), micro_f1=f1_score(G, PRED[m], average="micro", zero_division=0),
                               n_positive_labels=int(G.sum()), mean_labels_per_comment=G.sum(1).mean()))
agg_class = pd.DataFrame(rows)

# Add the class support under the simplified definition as a reference column
agg_class.insert(0, ("", "support_simplified"), GOLDS["simplified"].sum(0))
agg_macro = pd.DataFrame(macro_rows)

print("Macro/micro F1 under each gold definition:"); display(agg_macro.round(4))
print("Per-class F1 under each gold definition:"); display(agg_class.round(3))

save_table(agg_macro.set_index(["model", "gold"]), "06_aggregation_macro"); save_table(agg_class, "06_aggregation_per_class")
RES["agg_macro"], RES["agg_class"] = agg_macro, agg_class

# One panel per model; marker shape shows the gold definition
fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
order = names[:]
for a, m in zip(ax, MODELS):
    for gname, mk in zip(GOLDS, ["o", "s", "^", "D"]):
        a.scatter(agg_class[(SHORT[m], gname)].reindex(order), range(K), marker=mk, label=gname, s=22)
    a.set_yticks(range(K)); a.set_yticklabels(order, fontsize=7); a.set_title(SHORT[m]); a.set_xlabel("per-class F1"); a.grid(alpha=.3)
ax[0].legend(fontsize=7); fig.tight_layout(); save_fig(fig, "06_aggregation_sensitivity"); plt.show()

## Split by emotion, subreddit and comment length
### By emotion
Per-emotion F1 for both models next to the per-emotion agreement, plus the most common "miss X, predict Y" confusions inside the same comment.

In [ ]:
# Per-emotion table joining F1 with inter-annotator agreement
emo = per_class[["support", "F1_TF-IDF", "F1_Embedding"]].join(iaa[["prevalence", "raw_agreement", "pos_agreement", "kappa_pooled", "kappa_pairwise"]])
for m in MODELS:
    emo[f"recall_{SHORT[m]}"] = per_class[f"R_{SHORT[m]}"]
emo["best_model"] = np.where(emo["F1_Embedding"] > emo["F1_TF-IDF"], "Embedding", "TF-IDF")
display(emo.sort_values("F1_TF-IDF").round(3))
save_table(emo, "07_per_emotion_f1_and_agreement"); RES["emo"] = emo


def boot_over_emotions(x, y, n=N_BOOT, seed=SEED):
  '''
  Spearman correlation between two per-emotion vectors with a bootstrap CI that
  resamples emotions (not comments). Returns (rho, lower, upper).
  '''
    # Drop emotions where either value is missing
    x, y = np.asarray(x, float), np.asarray(y, float); ok = ~(np.isnan(x) | np.isnan(y)); x, y = x[ok], y[ok]
    r = np.random.default_rng(seed); rho = stats.spearmanr(x, y)[0]
    # Correlation on each resample of emotions
    bs = [stats.spearmanr(x[i], y[i])[0] for i in (r.integers(0, len(x), len(x)) for _ in range(n))]

    return float(rho), *ci95(bs)

rows = []

for m in MODELS:
    for agree_col in ("kappa_pooled", "pos_agreement", "raw_agreement"):
        rho, lo, hi = boot_over_emotions(emo[agree_col], emo[f"F1_{SHORT[m]}"])
        rows.append(dict(model=SHORT[m], agreement_measure=agree_col, spearman_with_F1=rho, ci_lo=lo, ci_hi=hi))

# Correlation between per-emotion agreement and F1, with CIs from resampling emotions
emo_corr = pd.DataFrame(rows); display(emo_corr.round(3)); save_table(emo_corr.set_index(["model", "agreement_measure"]), "07_agreement_vs_f1_correlation")
RES["emo_corr"] = emo_corr

# Scatter of agreement vs F1 per emotion; point size grows with test support
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for a, m in zip(ax, MODELS):
    a.scatter(emo["kappa_pooled"], emo[f"F1_{SHORT[m]}"], s=emo["support"] ** 0.5 * 3 + 10, color=COLORS[m], alpha=.7)
    for n_, r_ in emo.iterrows():
        a.annotate(n_, (r_["kappa_pooled"], r_[f"F1_{SHORT[m]}"]), fontsize=6.5, xytext=(2, 2), textcoords="offset points")
    a.set_xlabel("per-emotion inter-annotator agreement (pooled kappa)"); a.set_title(SHORT[m]); a.grid(alpha=.3)
ax[0].set_ylabel("test F1 for the emotion"); fig.tight_layout(); save_fig(fig, "07_agreement_vs_f1"); plt.show()

In [ ]:
# Most frequent "gold emotion missed, other emotion predicted instead (same comment)" pairs
# Most frequent 'gold emotion missed, other emotion predicted instead' pairs
conf_rows = []

for m in MODELS:
    # FN: gold labels the model missed. FPx: labels it predicted that are not gold
    FN = ((Yg == 1) & (PRED[m] == 0)).astype(int); FPx = ((Yg == 0) & (PRED[m] == 1)).astype(int)

    # M[i, j]: comments where gold emotion i was missed while emotion j was predicted
    M = FN.T @ FPx
    flat = [(names[i], names[j], int(M[i, j]), M[i, j] / max(FN[:, i].sum(), 1)) for i in range(K) for j in range(K) if i != j and M[i, j] > 0]
    top = sorted(flat, key=lambda t: -t[2])[:10]
    conf_rows += [dict(model=SHORT[m], gold_missed=a, predicted_instead=b, n=n_, share_of_misses_of_gold=s_) for a, b, n_, s_ in top]

confusions = pd.DataFrame(conf_rows)

display(confusions.round(3)); save_table(confusions.set_index(["model", "gold_missed"]), "07_top_confusions")

### By subreddit (largest subreddits in the test set as unit)
Reported per subreddit: error rate and micro-F1 with bootstrap CIs, the mean entropy of its comments, and the entropy-adjusted residual (observed error minus the error expected for comments of that entropy, using the pooled test set). A subreddit with a high error rate but a residual near zero is "hard" mostly because its comments are more ambiguous; a positive residual whose CI excludes zero is harder than its ambiguity explains.

In [ ]:
# The largest subreddits in the test set
top_subs = D["subreddit"].value_counts().head(TOP_SUBREDDITS).index.tolist()
for m in MODELS:
    # Residual error: wrong minus the error rate expected for comments in the same entropy decile
    D[f"resid_{m}"] = D[f"wrong_{m}"] - D.groupby("H_decile")[f"wrong_{m}"].transform("mean")


def sub_table(col, levels):
  '''
  Per-group error table for any grouping column (subreddit, comment length).
  For each group and model: error rate, micro-F1 and entropy-adjusted residual, each with a bootstrap CI,
  plus group size, mean entropy and share of split-bin comments.
  '''
    # One output row per group
    rows = []
    for lv in levels:
        # Comments belonging to this group
        mk = (D[col] == lv).to_numpy()
        r = dict(group=lv, n=int(mk.sum()), mean_entropy=D.loc[mk, ENTROPY_COL].mean(), share_split=(D.loc[mk, "agree_bin"] == "split").mean())
        for m in MODELS:
            pt, lo, hi = rate_ci(D[f"wrong_{m}"], mk)
            # Per-comment true positives, false positives and false negatives, used for micro-F1
            tp, fp, fn = (D[f"{k}_{m}"].to_numpy() for k in ("tp", "fp", "fn"))
            # Micro-F1 for the group from pooled counts
            mf = 2 * tp[mk].sum() / max(2 * tp[mk].sum() + fp[mk].sum() + fn[mk].sum(), 1)
            # Bootstrap CI for the group's micro-F1
            mlo, mhi = ci95(boot_micro_f1(tp, fp, fn, mk))
            # Entropy-adjusted residual with CI (positive = harder than its ambiguity explains)
            rpt, rlo, rhi = rate_ci(D[f"resid_{m}"], mk)
            r.update({f"err_{SHORT[m]}": f"{pt:.3f} [{lo:.3f}, {hi:.3f}]", f"microF1_{SHORT[m]}": f"{mf:.3f} [{mlo:.3f}, {mhi:.3f}]",
                      f"resid_{SHORT[m]}": f"{rpt:+.3f} [{rlo:+.3f}, {rhi:+.3f}]"})
        rows.append(r)
    return pd.DataFrame(rows).set_index("group")

# Per-subreddit results
sub_tbl = sub_table("subreddit", top_subs)
display(sub_tbl.round(3)); save_table(sub_tbl, "07_by_subreddit"); RES["sub"] = sub_tbl
overall_row = {SHORT[m]: D[f"wrong_{m}"].mean() for m in MODELS}
print("overall test error rate:", {k: round(v, 3) for k, v in overall_row.items()}, "| overall mean entropy:", round(D[ENTROPY_COL].mean(), 3))

### By comment length (words as unit)

In [ ]:
# Bucket comments by word count
D["len_bin"] = pd.cut(D["n_words"], [0, 5, 10, 15, 20, 10_000], labels=["1-5", "6-10", "11-15", "16-20", "21+"]).astype(str)
# Results by comment length
len_tbl = sub_table("len_bin", ["1-5", "6-10", "11-15", "16-20", "21+"])
display(len_tbl.round(3)); save_table(len_tbl, "07_by_comment_length"); RES["len"] = len_tbl

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
# Left: error by top subreddit. Right: error by comment length

for a, tbl, ttl in ((ax[0], sub_tbl, "Top subreddits"), (ax[1], len_tbl, "Comment length (words)")):
    for m in MODELS:
        vals = [float(s.split()[0]) for s in tbl[f"err_{SHORT[m]}"]]
        a.plot(range(len(tbl)), vals, marker="o", color=COLORS[m], label=SHORT[m])
    a.set_xticks(range(len(tbl))); a.set_xticklabels(tbl.index, rotation=60 if ttl.startswith("Top") else 0, ha="right" if ttl.startswith("Top") else "center", fontsize=8)
    a.set_ylabel("error rate"); a.set_title(ttl); a.legend()
fig.tight_layout(); save_fig(fig, "07_subreddit_and_length"); plt.show()

## Qualitative review (about 40 errors)
Sampling: Errors are comments where at least one model is wrong. They are stratified by agreement bin (3) and model pattern (`both_miss`, `only_tfidf_miss`, `only_emb_miss`), giving 9 cells. Slots are allocated round-robin so cells get near-equal counts, and a cell with fewer comments than its quota gives its remaining slots to the others. The draw is seeded.

Labelling protocol.
human check, where Piper and I check whether or not the model works
* `true_failure`: the gold label is clearly right and the model's output is clearly wrong
* `plausible_alternative`: the model's output is a defensible reading even though it differs from gold, note that multiple labels are counted as errors even when they may include the gold label.
* `ambiguous`: reasonable readers could not settle on a single reading (gold itself is questionable)

Then disagreements are resolved by discussion into the `resolved` column. The sheet is written once and **never overwritten** on re-runs, so labels are safe. The `hint` column (extra emotions a model predicted that exactly one rater chose) acts as a prompt for the reader not the label

In [ ]:
PATTERNS_ERR = ["both_miss", "only_tfidf_miss", "only_emb_miss"]
# Candidate errors: comments where at least one model is wrong
ERRS = D[D["pattern"].isin(PATTERNS_ERR)]
# One stratum per (agreement bin, model pattern)
cells_ = {(b, p): ERRS[(ERRS["agree_bin"] == b) & (ERRS["pattern"] == p)] for b in BINS for p in PATTERNS_ERR}


def allocate(sizes, total, seed):
  '''
  Split a total sample size across strata as evenly as possible.
  Adds one comment at a time round-robin; a stratum cannot exceed its size, and any leftover slots
  go to the other strata. Returns {stratum: number to draw}.
  '''
    # rng: seeded shuffler; q: slots given so far per stratum; rem: slots left to hand out
    rng, q, rem = np.random.default_rng(seed), {c: 0 for c in sizes}, total
    while rem > 0:
        # Strata that still have comments available
        open_ = [c for c in sizes if q[c] < sizes[c]]
        if not open_: break
        # Random order so ties are not always resolved in favour of the same strata
        rng.shuffle(open_)
        for c in open_:
            if rem == 0: break
            q[c] += 1; rem -= 1
    return q

# How many comments to draw from each stratum
quota = allocate({k: len(v) for k, v in cells_.items()}, SAMPLE_N, SEED)
print(pd.DataFrame({"available": {k: len(v) for k, v in cells_.items()}, "sampled": quota}).rename_axis(["agree_bin", "pattern"]))

# Map comment id to its row position in the test arrays
pos = {cid: i for i, cid in enumerate(test_ids)}
# Turn a 0/1 label vector into readable label names
lab_of = lambda v: ", ".join(names[j] for j in np.where(v)[0]) or "(none)"


def votes_str(i):
  '''
  Format one comment's rater votes as text, e.g. 'joy:3, love:1 (n=4)'.
  '''
    # Emotions in descending vote order
    o = np.argsort(-Vt[i], kind="stable"); return ", ".join(f"{names[j]}:{Vt[i, j]}" for j in o if Vt[i, j] > 0) + f" (n={nt[i]})"


def fp_hint(i):
  '''
  Hint text for the labelling sheet: extra emotions a model predicted that exactly one rater chose.
  '''
    # Collect one note per model that has extra emotions chosen by a single rater
    parts = []
    for m in MODELS:
        fp = [names[j] for j in np.where((PRED[m][i] == 1) & (Yg[i] == 0) & (Vt[i] >= 1))[0]]
        if fp: parts.append(f"{SHORT[m]} extra emotion chosen by 1 rater: {', '.join(fp)}")
    return " | ".join(parts)

# Draw the seeded sample from each stratum and order it for easy reading
sample = pd.concat([v.sample(n=quota[k], random_state=SEED) for k, v in cells_.items() if quota[k] > 0])
sample = sample.sort_values(["agree_bin", "pattern"]).reset_index(drop=True)

# Row positions of the sampled comments in the test arrays
ii = sample["comment_id"].map(pos).to_numpy()

# Build the labelling sheet; the label columns start empty
qual = pd.DataFrame({"comment_id": sample["comment_id"], "subreddit": sample["subreddit"], "text": sample["text"],
                     "agree_bin": sample["agree_bin"].astype(str), "entropy": sample[ENTROPY_COL].round(3), "pattern": sample["pattern"],
                     "gold": [lab_of(Yg[i]) for i in ii], "rater_votes": [votes_str(i) for i in ii],
                     "pred_tfidf": [lab_of(PRED["tfidf"][i]) for i in ii], "pred_emb": [lab_of(PRED["emb"][i]) for i in ii],
                     "hint": [fp_hint(i) for i in ii],
                     "label_helen": "", "label_piper": "", "resolved": "", "notes": ""})

QUAL_PATH = f"{OUT}/qualitative_review_sample.csv"

# Never overwrite an existing sheet, so completed labels are safe
if os.path.exists(QUAL_PATH):
    print("Existing labelling sheet found, NOT overwriting:", QUAL_PATH)
else:
    qual.to_csv(QUAL_PATH, index=False); print("Wrote labelling sheet:", QUAL_PATH)
display(pd.read_csv(QUAL_PATH).drop(columns=["label_helen", "label_piper", "resolved", "notes"]).head(12))

### ***Post human-check/assessment, reporting inter labeller agreement***
the CSV will appear in OUT, the completed CSV has been retained for our purposes


In [ ]:
# The three allowed labels
QUALDONE_PATH  = f"{OUT}/qualitative_review_sample_completed.csv"

LABELS = ["true_failure", "plausible_alternative", "ambiguous"]
# Read the (possibly labelled) sheet and normalise the label text
q = pd.read_csv(QUALDONE_PATH)
for c in ("label_helen", "label_piper", "resolved"):
    q[c] = q[c].astype("string").str.strip().str.lower()

# Flag any label that is not one of the three allowed values
bad = q[["label_helen", "label_piper", "resolved"]].stack().dropna()
bad = bad[~bad.isin(LABELS)]

if len(bad): print("WARNING: labels outside", LABELS, "->", bad.unique().tolist())


# Rows that both have labelled
both = q.dropna(subset=["label_helen", "label_piper"])

if len(both) == 0:
    print("Labels not filled, fill labels in .csv", QUAL_PATH)
else:
    print(f"Both labelled: {len(both)}/{len(q)} | raw agreement {(both.label_helen == both.label_piper).mean():.2f} | "
          f"Cohen's kappa {cohen_kappa_score(both.label_helen, both.label_piper):.2f}")
    display(pd.crosstab(both.label_helen, both.label_piper))
    dis = both[both.label_helen != both.label_piper]
    print(f"{len(dis)} disagreements to resolve:"); display(dis[["comment_id", "text", "gold", "rater_votes", "pred_tfidf", "pred_emb", "label_helen", "label_piper", "resolved"]])

    # Final label: the resolved value if present, otherwise the shared label when both agree
    final = q.copy()
    final["final"] = final["resolved"].fillna(final["label_helen"].where(final["label_helen"] == final["label_piper"]))
    fin = final.dropna(subset=["final"])
    if len(fin) < len(final):
        print(f"NOTE: {len(final) - len(fin)} rows still have no final label (unresolved disagreements or unlabelled).")
    if len(fin):
        # Resolved labels by agreement bin (counts and row shares)
        ct1 = pd.crosstab(fin["agree_bin"], fin["final"]).reindex(index=BINS, columns=LABELS, fill_value=0)
        display(ct1); display(ct1.div(ct1.sum(1).replace(0, np.nan), axis=0).round(2))
        display(pd.crosstab(fin["pattern"], fin["final"]).reindex(index=PATTERNS_ERR, columns=LABELS, fill_value=0))
        save_table(ct1, "08_resolved_labels_by_agreement_bin"); RES["qual"] = ct1

#Code for write up
* annotated examples: prints a few comments from selected cells with the rater votes and both models' outputs, so the report can quote concrete cases for "true failure vs. plausible alternative vs. ambiguous".

* Assembles the **model comparison table** and prints the headline numbers. All tables are also saved in `OUT/eval_tables` and all figures in `OUT/eval_figs`.

In [ ]:
def show_examples(pattern, agree_bin, k=2, seed=SEED):
  '''
  Print k random test comments from one (model pattern, agreement bin) cell with their
  rater votes and both models' predictions, for quoting in the write-up.
  '''
    # Test comments in this (pattern, bin) cell
    sub = D[(D["pattern"] == pattern) & (D["agree_bin"] == agree_bin)]
    if len(sub) == 0:
        print(f"[{pattern} / {agree_bin}] no comments"); return
    print(f"\n=== {pattern} | {agree_bin} bin | {len(sub)} comments in test ===")
    # Random examples, seeded for reproducibility
    for _, r in sub.sample(n=min(k, len(sub)), random_state=seed).iterrows():
        i = pos[r["comment_id"]]
        print(f"- r/{r['subreddit']}: {r['text']!r}\n    gold: {lab_of(Yg[i])} | raters: {votes_str(i)}\n"
              f"    TF-IDF: {lab_of(PRED['tfidf'][i])} | Embedding: {lab_of(PRED['emb'][i])}")
# A few cells to quote as examples in the write-up
for p, b in [("both_miss", "unanimous"), ("both_miss", "split"), ("only_tfidf_miss", "majority"), ("only_emb_miss", "majority")]:
    show_examples(p, b)

In [ ]:
# One row per model, pulling from the results registry built in earlier sections
rows = {}

for m in MODELS:
    ebin = RES["err_by_bin"][RES["err_by_bin"].model == SHORT[m]].set_index("agree_bin")
    cr = RES["corr"][(RES["corr"].model == SHORT[m]) & (RES["corr"].entropy == ENTROPY_COL) & (RES["corr"].error == "err_f1")].iloc[0]
    pl_ = RES["plaus"][(RES["plaus"].model == SHORT[m]) & (RES["plaus"].agree_bin == "all")].iloc[0]
    rows[f"{SHORT[m]} ({MODELS[m]})"] = {
        "macro_F1": RES["std"].iloc[list(MODELS).index(m)]["macro_f1"], "micro_F1": RES["std"].iloc[list(MODELS).index(m)]["micro_f1"],
        "hamming": RES["std"].iloc[list(MODELS).index(m)]["hamming"], "error_all": D[f"wrong_{m}"].mean(),
        **{f"error_{b}": ebin.loc[b, "error_rate"] for b in BINS},
        "spearman(entropy, 1-F1)": f"{cr.spearman:+.2f} [{cr.ci_lo:+.2f}, {cr.ci_hi:+.2f}]",
        "mean_JSD": D[f"jsd_{m}"].mean(),
        "errors_with_extra_emotion_chosen_by_1_rater": pl_["fp_one_rater"],
        "errors_omission_only": pl_["omission_only"]}

# Final model comparison table
model_table = pd.DataFrame(rows).T
display(model_table.round(3)); save_table(model_table, "09_model_comparison_table"); RES["model_table"] = model_table

In [ ]:
# Print the headline numbers with CIs for filling in the write-up
print("HEADLINE NUMBERS (fill the write-up from these; all test set, 95% bootstrap CIs)\n")
for m in MODELS:
    e = RES["err_by_bin"][RES["err_by_bin"].model == SHORT[m]].set_index("agree_bin")
    cr = RES["corr"][(RES["corr"].model == SHORT[m]) & (RES["corr"].entropy == ENTROPY_COL) & (RES["corr"].error == "err_f1")].iloc[0]
    print(f"{SHORT[m]:9s} error: " + " | ".join(f"{b} {e.loc[b, 'error_rate']:.3f} [{e.loc[b, 'er_lo']:.3f}, {e.loc[b, 'er_hi']:.3f}] (n={int(e.loc[b, 'n'])})" for b in BINS)
          + f"\n{'':9s} Spearman(entropy, 1-F1) = {cr.spearman:+.3f} [{cr.ci_lo:+.3f}, {cr.ci_hi:+.3f}]")

it = RES["interaction"]

print(f"\nModel comparison: overall Delta (TF-IDF err - Emb err) = {it['overall']:+.4f} [{it['overall_lo']:+.4f}, {it['overall_hi']:+.4f}]; "
      f"Delta(unanimous) - Delta(split) = {it['est']:+.4f} [{it['lo']:+.4f}, {it['hi']:+.4f}]")
print("Pattern counts:", D["pattern"].value_counts().to_dict())
print(f"\nSaved tables -> {TAB}\nSaved figures -> {FIG}")
print(sorted(os.listdir(FIG)))